# 07 - Quantile yield models and honest prediction intervals

**Kenya maize (One Acre Fund MEL Agronomic Survey, 2016-2020)**

Notebooks 04-06 focused on point predictions and model selection. This notebook adds the missing deployment question: how uncertain is a plot-level yield forecast? It trains LightGBM quantile models for the 10th, 50th, and 90th percentiles, then calibrates a split-conformal interval using the 2019 season only. The 2020 season remains a genuinely unseen holdout.

The interval is useful only if its empirical coverage is close to its nominal 80% level. Coverage and width are reported overall and by district; a narrow interval with poor coverage is not a useful forecast.

## 0 - Setup and evaluation protocol

In [ ]:
import os, warnings
from pathlib import Path

os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count() or 4))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
pd.set_option('display.width', 180)

ROOT = Path.cwd()
while not (ROOT / 'data' / 'features').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
FEATURES = ROOT / 'data' / 'features'
OUTPUTS = ROOT / 'outputs' / 'modeling'
OUTPUTS.mkdir(parents=True, exist_ok=True)
TARGET, HOLDOUT_YEAR, CALIBRATION_YEAR = 'yield_kg_ph', 2020, 2019
LOW_Q, MID_Q, HIGH_Q = 0.10, 0.50, 0.90
print(f'LightGBM {lgb.__version__}')
print(f'Feature directory: {FEATURES}')

In [ ]:
def restore_dtypes(df):
    for column in df.columns:
        if df[column].dtype == bool:
            df[column] = df[column].astype(float)
        elif df[column].dtype == object:
            values = set(df[column].dropna().unique())
            if values and values <= {True, False, 'True', 'False'}:
                df[column] = df[column].map({True: 1.0, False: 0.0, 'True': 1.0, 'False': 0.0}).astype(float)
    return df

full = restore_dtypes(pd.read_csv(FEATURES / 'kenya_maize_features_full.csv', low_memory=False))
recommended = pd.read_csv(FEATURES / 'kenya_maize_recommended_features.csv')
FEATURES_USED = recommended.feature.tolist()
CATEGORICAL = [column for column in FEATURES_USED if full[column].dtype == object]
for column in CATEGORICAL:
    full[column] = full[column].astype('category')

train_mask = full.year < HOLDOUT_YEAR
calibration_mask = full.year == CALIBRATION_YEAR
holdout_mask = full.year == HOLDOUT_YEAR
X = full[FEATURES_USED]
y = full[TARGET]
print(f'{len(FEATURES_USED)} features, {len(CATEGORICAL)} categorical')
print(f'train: {train_mask.sum():,} rows | calibration: {calibration_mask.sum():,} | holdout: {holdout_mask.sum():,}')

## 1 - Quantile models

Each model estimates a conditional quantile rather than the conditional mean. The 10th and 90th percentile models form a nominal 80% forecast band; the median is also used as a robust point forecast. Hyperparameters are fixed from the earlier modeling work, so the 2020 holdout is not used for selection.

In [ ]:
def make_quantile_model(alpha):
    return lgb.LGBMRegressor(
        objective='quantile', alpha=alpha,
        n_estimators=350, learning_rate=0.035, num_leaves=20,
        min_child_samples=35, subsample=0.85, colsample_bytree=0.85,
        reg_lambda=2.0, random_state=42, verbosity=-1
    )

def fit_quantiles(fit_mask):
    models, predictions = {}, {}
    for quantile in (LOW_Q, MID_Q, HIGH_Q):
        model = make_quantile_model(quantile)
        model.fit(X.loc[fit_mask], y.loc[fit_mask], categorical_feature=CATEGORICAL)
        models[quantile] = model
        predictions[quantile] = model.predict(X)
    return models, pd.DataFrame(predictions, index=full.index).rename(columns={LOW_Q: 'q10', MID_Q: 'q50', HIGH_Q: 'q90'})

final_models, final_predictions = fit_quantiles(train_mask)
print('Final quantile models fitted.')

In [ ]:
def point_metrics(actual, predicted):
    return {'r2': r2_score(actual, predicted), 'mae': mean_absolute_error(actual, predicted), 'rmse': float(np.sqrt(np.mean((actual - predicted) ** 2)))}

holdout_y = y.loc[holdout_mask].to_numpy()
holdout_pred = final_predictions.loc[holdout_mask]
district_mean = full.loc[train_mask].groupby('district', observed=True)[TARGET].mean()
district_baseline = full.loc[holdout_mask, 'district'].map(district_mean).fillna(y.loc[train_mask].mean()).to_numpy()
point_results = pd.DataFrame([
    {'model': 'training global mean', **point_metrics(holdout_y, np.full(len(holdout_y), y.loc[train_mask].mean()))},
    {'model': 'training district mean', **point_metrics(holdout_y, district_baseline)},
    {'model': 'quantile median', **point_metrics(holdout_y, holdout_pred.q50.to_numpy())},
])
display(point_results.round(4))

## 2 - Split-conformal calibration

The raw q10-q90 interval may be miscalibrated. Models are fitted through 2018, evaluated on 2019, and the largest miss outside the raw interval is measured for every calibration row. The 90th percentile of those nonconformity scores is then added to both sides of the final 2020 interval. This uses no 2020 outcomes.

In [ ]:
pre_calibration_mask = full.year < CALIBRATION_YEAR
calibration_models, calibration_predictions = fit_quantiles(pre_calibration_mask)
calibration_lower = calibration_predictions.loc[calibration_mask, 'q10'].to_numpy()
calibration_upper = calibration_predictions.loc[calibration_mask, 'q90'].to_numpy()
calibration_actual = y.loc[calibration_mask].to_numpy()
nonconformity = np.maximum(calibration_lower - calibration_actual, calibration_actual - calibration_upper)
conformal_adjustment = float(np.quantile(nonconformity, 0.90, method='higher'))
raw_lower = holdout_pred.q10.to_numpy()
raw_upper = holdout_pred.q90.to_numpy()
interval_lower = raw_lower - conformal_adjustment
interval_upper = raw_upper + conformal_adjustment
actual = holdout_y
raw_coverage = np.mean((actual >= raw_lower) & (actual <= raw_upper))
conformal_coverage = np.mean((actual >= interval_lower) & (actual <= interval_upper))
print(f'2019 conformal adjustment: {conformal_adjustment:,.0f} kg/ha')
print(f'2020 raw 80% interval coverage: {raw_coverage:.1%}')
print(f'2020 conformal interval coverage: {conformal_coverage:.1%}')
print(f'2020 conformal mean width: {(interval_upper - interval_lower).mean():,.0f} kg/ha')

In [ ]:
prediction_output = full.loc[holdout_mask, ['year', 'district']].copy()
prediction_output['actual_yield_kg_ph'] = actual
prediction_output['predicted_q10_kg_ph'] = raw_lower
prediction_output['predicted_q50_kg_ph'] = holdout_pred.q50.to_numpy()
prediction_output['predicted_q90_kg_ph'] = raw_upper
prediction_output['conformal_lower_kg_ph'] = interval_lower
prediction_output['conformal_upper_kg_ph'] = interval_upper
prediction_output['covered_by_conformal_interval'] = (actual >= interval_lower) & (actual <= interval_upper)
district_results = (prediction_output.groupby('district', observed=True)
    .agg(n=('actual_yield_kg_ph', 'size'), coverage=('covered_by_conformal_interval', 'mean'),
         mean_width=('conformal_upper_kg_ph', lambda values: np.mean(values - prediction_output.loc[values.index, 'conformal_lower_kg_ph'])),
         actual_mean=('actual_yield_kg_ph', 'mean'), predicted_median=('predicted_q50_kg_ph', 'mean'))
    .query('n >= 20').sort_values('coverage'))
display(district_results.head(10).round(3))
point_results.to_csv(OUTPUTS / 'quantile_point_metrics.csv', index=False)
prediction_output.to_csv(OUTPUTS / 'quantile_2020_predictions.csv', index=False)
district_results.to_csv(OUTPUTS / 'quantile_district_interval_metrics.csv')
print(f'Saved results under {OUTPUTS}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sample = prediction_output.sort_values('actual_yield_kg_ph').reset_index(drop=True)
sample = sample.iloc[::max(1, len(sample) // 250)]
axes[0].fill_between(np.arange(len(sample)), sample.conformal_lower_kg_ph, sample.conformal_upper_kg_ph, color='#9ecae1', alpha=0.8, label='conformal 80% interval')
axes[0].plot(sample.index, sample.actual_yield_kg_ph, '.', color='#222222', ms=3, label='actual')
axes[0].plot(sample.index, sample.predicted_q50_kg_ph, color='#e6550d', lw=1.5, label='median')
axes[0].set_title('2020 predictions ordered by actual yield', loc='left')
axes[0].set_xlabel('holdout plots, sorted sample')
axes[0].set_ylabel('yield (kg/ha)')
axes[0].legend(frameon=False)
coverage_by_district = district_results.sort_values('coverage')
axes[1].barh(coverage_by_district.index.astype(str), coverage_by_district.coverage, color='#31a354')
axes[1].axvline(0.8, color='#e6550d', ls='--', lw=1.5)
axes[1].set_xlim(0, 1)
axes[1].set_title('District coverage (districts with >=20 plots)', loc='left')
axes[1].set_xlabel('share covered by conformal interval')
fig.tight_layout()
plt.show()

In [ ]:
crossing_rate = np.mean((holdout_pred.q10 > holdout_pred.q50) | (holdout_pred.q50 > holdout_pred.q90))
print(f'Quantile crossing rate on 2020: {crossing_rate:.2%}')
print(f'Raw interval mean width: {(raw_upper - raw_lower).mean():,.0f} kg/ha')
print(f'Conformal interval mean width: {(interval_upper - interval_lower).mean():,.0f} kg/ha')
print('Interpret intervals as population-level coverage targets, not guaranteed plot-specific probabilities.')

## Conclusion

The median quantile model provides a robust point forecast, while the conformal interval makes uncertainty visible and testable. The key outputs are the 2020 coverage, interval width, and district-level variation in coverage. If coverage is materially below 80%, the interval should not be presented as calibrated; if it is close to 80% but very wide, the data supports cautious ranking more than precise plot-level intervention.

This notebook does not change the plot-level ceiling established in notebooks 05-06. It changes the reporting contract: every forecast should carry an uncertainty range and a clear statement of what that range has been validated to cover.